# 02 · Thinking in N dimensions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/02-thinking-in-n-dimensions.ipynb)

*Part II · group · 20 min*

> 🇪🇸 **Pensar en N dimensiones** — Aprender a leer tensores reales preguntando qué cuenta cada eje y por qué un eje de lote no significa lo mismo que un eje temporal.

Learn to read real tensors by asking what every axis counts and why a batch axis is not the same thing as a time axis.

## What you will be able to do

- Read the order and shape of real image and video tensors and explain what each axis counts.
- Compare two real tensors with the same shape but different axis semantics.
- Show with real data why shuffling a batch can be valid while shuffling time changes the data's meaning.
- Build a padded order-5 batch from real video clips of different lengths and carry a validity mask.

## Setup

Run this first. It loads real image and video data used throughout the notebook:

1. handwritten digit images from `sklearn.datasets.load_digits`,
2. a real RGB photograph from `skimage.data.astronaut`, and
3. a pinned CC0 video from Wikimedia Commons, the same verified clip used later in section 05.

> 🇪🇸 Ejecuta esta celda primero. Carga datos reales de imágenes y video: dígitos manuscritos, una fotografía RGB y un video CC0 verificado de Wikimedia Commons.

In [1]:
%pip install -q "imageio[ffmpeg]"

import hashlib
import io
import urllib.request

import imageio.v3 as iio
import numpy as np
from sklearn.datasets import load_digits
from skimage import data

rng = np.random.default_rng(0)

# ---------------------------------------------------------------------------
# Real image data: handwritten digits
# ---------------------------------------------------------------------------
digits = load_digits()
digit_batch = digits.images[:8].astype(np.float32)   # (N, H, W)
digit_labels = digits.target[:8]
real_digit = digit_batch[0]
real_photo = data.astronaut()                        # real RGB photograph, (H, W, C)

# ---------------------------------------------------------------------------
# Real video data: "Tormenta en l'Almadrava" by Nicolas Vigier, CC0
# Same pinned source/checksum already used by notebook 05.
# ---------------------------------------------------------------------------
VIDEO_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1e/"
    "Tormenta_en_l%27Almadrava.webm"
)
VIDEO_SHA256 = "e377fcdd2c79b55bce13c2c24b5dd7e412af39cd400eec548a79d0e59d79dc1b"
UA = "tensors-workshop/1.0 (https://github.com/project-delphi/tensors-workshop)"


def fetch_verified_video(url, expected_sha256, n_frames=16, stride=45):
    """Download, checksum, and retain sampled frames from a real video."""
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    raw = urllib.request.urlopen(req, timeout=120).read()

    got = hashlib.sha256(raw).hexdigest()
    if got != expected_sha256:
        raise ValueError(
            f"checksum mismatch: expected {expected_sha256}, got {got}"
        )

    frames = []
    for i, frame in enumerate(
        iio.imiter(io.BytesIO(raw), plugin="FFMPEG", extension=".webm")
    ):
        if i % stride == 0:
            frames.append(frame)
            if len(frames) == n_frames:
                break

    return np.stack(frames)


real_video = fetch_verified_video(VIDEO_URL, VIDEO_SHA256)
assert real_video.shape == (16, 540, 960, 3), real_video.shape

# Build a small REAL temporal tensor with exactly the same shape as digit_batch:
# (8, 8, 8). We take 8 real video frames, a centered 8x8 crop, and average RGB.
r0 = real_video.shape[1] // 2 - 4
c0 = real_video.shape[2] // 2 - 4
video_patch = real_video[:8, r0:r0 + 8, c0:c0 + 8].mean(axis=3).astype(np.float32)

print("real_digit :", real_digit.shape, real_digit.dtype)
print("digit_batch:", digit_batch.shape, digit_batch.dtype)
print("real_photo :", real_photo.shape, real_photo.dtype)
print("real_video :", real_video.shape, real_video.dtype)
print("video_patch:", video_patch.shape, video_patch.dtype)


real_digit : (8, 8) float32
digit_batch: (8, 8, 8) float32
real_photo : (512, 512, 3) uint8
real_video : (16, 540, 960, 3) uint8
video_patch: (8, 8, 8) float32


## Why this matters

A tensor shape is only the beginning. The same tuple of integers can describe completely different experiments.

In this notebook, `digit_batch` and `video_patch` both have shape **`(8, 8, 8)`**:

- `digit_batch`: `(N, H, W)` — 8 independent handwritten-digit images.
- `video_patch`: `(T, H, W)` — 8 ordered moments from a real video.

The arrays have the same order and the same shape. Their **axis 0 does not mean the same thing**.

> 🇪🇸 **Por qué importa:** `digit_batch` y `video_patch` tienen exactamente la misma forma `(8, 8, 8)`, pero en uno el eje 0 cuenta ejemplos independientes y en el otro cuenta instantes ordenados. La forma no contiene por sí sola esa semántica.

### Predict → Run → Explain

Before each code cell, predict what every axis counts. Then run the code and explain whether changing the order of an axis changes the meaning of the data.

> 🇪🇸 **Predice → Ejecuta → Explica:** antes de ejecutar, di qué cuenta cada eje. Después explica si cambiar su orden modifica o no el significado de los datos.

## 2.1 Real tensors can have different orders

We will not invent arrays with `np.zeros` to build a shape ladder. Instead, inspect real objects that already occur in data work:

| Real object | Shape | Order | Axis meaning |
|---|---:|---:|---|
| one handwritten digit | `(8, 8)` | 2 | `(H, W)` |
| batch of handwritten digits | `(8, 8, 8)` | 3 | `(N, H, W)` |
| RGB photograph | `(512, 512, 3)` | 3 | `(H, W, C)` |
| sampled real video | `(16, 540, 960, 3)` | 4 | `(T, H, W, C)` |

Later we will combine several real clips into an order-5 padded batch `(N, T, H, W, C)`.

> 🇪🇸 No construiremos la progresión con arreglos vacíos. Leeremos objetos reales: un dígito, un lote de dígitos, una fotografía RGB y un video. Al final construiremos un lote de videos de orden 5.

## Exercise 1 — read the real shapes

**Predict first.** For each object below, write:

1. its expected order,
2. what every axis counts, and
3. which axes could be shuffled without changing the meaning of the individual observations.

> 🇪🇸 **Predice primero.** Para cada objeto real, escribe su orden, qué cuenta cada eje y cuáles ejes podrían reorganizarse sin cambiar el significado de las observaciones individuales.

In [2]:
# TODO 1:
# Inspect these REAL tensors:
#
#   real_digit
#   digit_batch
#   real_photo
#   real_video
#
# For each one:
# 1. print .shape and .ndim
# 2. write a comment naming every axis
# 3. state whether reordering axis 0 preserves or changes its meaning
#
# Write your code below this line.


In [3]:
#@title Solution — try it yourself first { display-mode: 'form' }

objects = [
    ("one real digit", real_digit, "(H, W)"),
    ("real digit batch", digit_batch, "(N, H, W)"),
    ("real RGB photo", real_photo, "(H, W, C)"),
    ("real sampled video", real_video, "(T, H, W, C)"),
]

for name, arr, axes in objects:
    print(f"{name:20s} shape={str(arr.shape):20s} order={arr.ndim} axes={axes}")

print()
print("Axis 0 meaning:")
print("- real_digit : image rows; reordering them scrambles the image")
print("- digit_batch: independent examples; batch order can be changed")
print("- real_photo : image rows; reordering them scrambles the image")
print("- real_video : time; reordering it changes temporal meaning")


one real digit       shape=(8, 8)               order=2 axes=(H, W)
real digit batch     shape=(8, 8, 8)            order=3 axes=(N, H, W)
real RGB photo       shape=(512, 512, 3)        order=3 axes=(H, W, C)
real sampled video   shape=(16, 540, 960, 3)    order=4 axes=(T, H, W, C)

Axis 0 meaning:
- real_digit : image rows; reordering them scrambles the image
- digit_batch: independent examples; batch order can be changed
- real_photo : image rows; reordering them scrambles the image
- real_video : time; reordering it changes temporal meaning


<details>
<summary><strong>Why this solution works · Por qué funciona esta solución</strong></summary>

The number of axes tells us the **order**, but the dataset tells us what those axes **mean**. A batch axis is a collection of independent observations; a spatial or temporal axis carries internal structure.

> 🇪🇸 El número de ejes determina el **orden**, pero el conjunto de datos determina qué **significan** esos ejes. Un eje de lote reúne observaciones independientes; los ejes espaciales y temporales contienen estructura interna.

</details>

## 2.2 Same shape, different meaning

Now compare two **real** tensors with exactly the same shape:

```text
digit_batch.shape == (8, 8, 8)   # (N, H, W)
video_patch.shape == (8, 8, 8)   # (T, H, W)
```

For the digit batch, keeping each image paired with its label is what matters. The order of examples in the batch is not part of a digit's identity.

For the video tensor, axis 0 is time. Consecutive frames are related because the scene evolves from one moment to the next.

> 🇪🇸 Misma forma, significado diferente: en el lote de dígitos el eje 0 cuenta ejemplos independientes; en el video cuenta tiempo. El código ve enteros, pero el científico debe conservar la semántica.

## Exercise 2 — shuffle batch vs. shuffle time on real data

Use one permutation for both real tensors.

For the digit batch, shuffle **images and labels together**. For the video, shuffle the temporal axis. Then compare a simple temporal-continuity statistic before and after the shuffle.

Because the video was sampled with `stride=45`, these are **consecutive sampled frames**, not consecutive frames from the original video stream.

> 🇪🇸 Usa la misma permutación en ambos tensores. En el lote de dígitos reorganiza imágenes y etiquetas juntas. En el video reorganiza el eje temporal y compara una medida simple de continuidad antes y después. Como el video fue muestreado con `stride=45`, trabajamos con **fotogramas muestreados consecutivos**, no con fotogramas consecutivos del video original.

In [4]:
# TODO 2:
# 1. Create perm = rng.permutation(8).
# 2. Apply it to digit_batch AND digit_labels.
# 3. Apply it to video_patch.
# 4. Print original vs shuffled labels.
# 5. Compute the mean absolute change between consecutive SAMPLED video frames
#    before and after shuffling.
# 6. Explain why the digit batch still represents the same 8 labeled examples,
#    while the temporal story of the sampled video sequence has changed.
#
# Write your code below this line.


In [5]:
#@title Solution — try it yourself first { display-mode: 'form' }

perm = rng.permutation(8)

shuffled_digits = digit_batch[perm]
shuffled_labels = digit_labels[perm]
shuffled_video = video_patch[perm]

print("original digit labels:", digit_labels)
print("shuffled digit labels:", shuffled_labels)
print("same labeled examples? ", sorted(zip(digit_labels.tolist(), digit_batch.sum(axis=(1, 2)).round(6).tolist()))
      == sorted(zip(shuffled_labels.tolist(), shuffled_digits.sum(axis=(1, 2)).round(6).tolist())))


def mean_consecutive_sampled_change(x):
    x = x.astype(np.float32)
    return float(np.mean(np.abs(x[1:] - x[:-1])))


before = mean_consecutive_sampled_change(video_patch)
after = mean_consecutive_sampled_change(shuffled_video)

print()
print(f"video mean consecutive sampled-frame change before shuffle: {before:.3f}")
print(f"video mean consecutive sampled-frame change after  shuffle: {after:.3f}")
print(f"after/before ratio: {after / before:.2f}x")

# Batch: the order of independent examples changed, but image-label pairs stayed intact.
# Time: the same sampled frames remain, but their temporal order no longer describes
# the original measured sequence.


original digit labels: [0 1 2 3 4 5 6 7]
shuffled digit labels: [2 4 3 6 5 0 1 7]
same labeled examples?  True

video mean consecutive sampled-frame change before shuffle: 18.365
video mean consecutive sampled-frame change after  shuffle: 47.814
after/before ratio: 2.60x


<details>
<summary><strong>What did the shuffle prove? · ¿Qué demostró la permutación?</strong></summary>

For the digits, the permutation changes **presentation order**, not the identity of the eight labeled observations. For the video, the permutation changes the measured chronology of the **sampled frame sequence**.

The continuity statistic is not a universal definition of "video correctness"; it is simply observable evidence that reordering real sampled frames changes their temporal relationships.

> 🇪🇸 En los dígitos cambia el **orden de presentación**, no la identidad de las ocho observaciones etiquetadas. En el video cambia la cronología medida de la **secuencia de fotogramas muestreados**. La estadística de continuidad es evidencia observable de que reorganizar fotogramas reales muestreados cambia sus relaciones temporales.

</details>

## 2.3 Real videos have different lengths

A batch needs one rectangular tensor, but real clips may contain different numbers of frames.

We will create three clips by taking three **different measured segments** from the real storm video. Their pixel values are real; only the segment boundaries are chosen for this teaching example.

For efficiency, we spatially subsample the frames before batching. Spatial subsampling keeps measured pixels but retains fewer of them.

> 🇪🇸 Los tres clips provienen de segmentos distintos del video real. Los valores de los píxeles son medidos; solo elegimos los límites de cada segmento con fines pedagógicos. Para ahorrar memoria conservamos uno de cada cuatro píxeles en cada dirección espacial.

## Exercise 3 — build an order-5 batch from real clips

Create three real clips with lengths `4`, `7`, and `5` frames. Pad them to the longest length and build a Boolean mask telling the model which frame slots contain measured data.

Predict the final tensor shape before running the solution.

> 🇪🇸 Construye tres clips reales de 4, 7 y 5 fotogramas. Rellénalos hasta la longitud máxima y crea una máscara booleana que indique qué posiciones contienen datos medidos. Predice primero la forma final.

In [6]:
# TODO 3:
# Work from real_video.
#
# 1. Spatially subsample it with real_video[:, ::4, ::4, :].
# 2. Take three non-overlapping real segments with lengths 4, 7, and 5.
# 3. Compute T_max.
# 4. Allocate one padded batch with shape (N, T_max, H, W, C).
# 5. Build a Boolean mask with shape (N, T_max).
# 6. Count how many frame slots are padding rather than measured frames.
#
# Write your code below this line.


In [7]:
#@title Solution — try it yourself first { display-mode: 'form' }

video_small = real_video[:, ::4, ::4, :]  # measured pixels, spatially subsampled

real_clips = [
    video_small[0:4],    # 4 measured frames
    video_small[4:11],   # 7 measured frames
    video_small[11:16],  # 5 measured frames
]

lengths = np.array([len(x) for x in real_clips])
T_max = int(lengths.max())
N = len(real_clips)
H, W, C = video_small.shape[1:]

padded = np.zeros((N, T_max, H, W, C), dtype=video_small.dtype)
valid = np.zeros((N, T_max), dtype=bool)

for n, x in enumerate(real_clips):
    T = len(x)
    padded[n, :T] = x
    valid[n, :T] = True

padded_slots = int((~valid).sum())
total_slots = int(valid.size)

print("real clip lengths:", lengths.tolist())
print("padded batch shape:", padded.shape)
print("batch order:", padded.ndim)
print("axes: (N, T, H, W, C)")
print("validity mask shape:", valid.shape)
print("measured frame slots:", int(valid.sum()))
print("padding frame slots:", padded_slots)
print(f"padding fraction: {padded_slots / total_slots:.1%}")

assert padded.shape == (3, 7, 135, 240, 3)
assert valid.sum() == 16


real clip lengths: [4, 7, 5]
padded batch shape: (3, 7, 135, 240, 3)
batch order: 5
axes: (N, T, H, W, C)
validity mask shape: (3, 7)
measured frame slots: 16
padding frame slots: 5
padding fraction: 23.8%


<details>
<summary><strong>Why padding needs a mask · Por qué el padding necesita una máscara</strong></summary>

The order-5 tensor is rectangular, but not every `(N, T)` location represents a recorded frame. The mask distinguishes **measured frames** from **padding values introduced by preprocessing**.

This is an important distinction: the underlying dataset is real, while padding is an explicit computational convention. Without the mask, zeros could be mistaken for observations.

> 🇪🇸 El tensor de orden 5 es rectangular, pero no toda posición `(N, T)` corresponde a un fotograma grabado. La máscara distingue **datos medidos** de **valores de relleno introducidos por el preprocesamiento**.

</details>

## 2.4 From video axes to experimental axes

The same reasoning applies to scientific data.

Suppose a microscope records cells repeatedly:

- `T` can count acquisition times.
- `H, W` can count pixel locations inside each image.
- `C` can count imaging channels.
- another axis can count fields of view, wells, dishes, patients, or experimental conditions.

A **field of view is not automatically the same thing as `H` or `W`**. `H` and `W` are pixel coordinates *inside* an image; multiple fields of view usually require their own observation axis or are organized into the batch structure.

> 🇪🇸 En microscopía, un **campo de visión no es automáticamente un eje `H` o `W`**. `H` y `W` describen coordenadas de píxeles dentro de una imagen; varios campos de visión suelen requerir otro eje de observación.

## What just happened

You worked with real measured image/video values throughout the core examples. The only introduced values were explicit padding values, tracked by a validity mask.

- a real handwritten digit gave an order-2 tensor `(H, W)`;
- eight real digits gave an order-3 batch `(N, H, W)`;
- a real RGB photograph gave an order-3 tensor `(H, W, C)`;
- a real sampled video gave an order-4 tensor `(T, H, W, C)`;
- three real video segments plus explicit padding produced an order-5 batch `(N, T, H, W, C)` and a validity mask.

The central lesson is not "higher order means more complicated." It is:

> **Every axis must have a meaning, and operations are only valid when they respect that meaning.**

Two tensors can have the same shape and still represent fundamentally different data.

> 🇪🇸 **Qué ocurrió:** trabajaste con valores reales medidos de imágenes y video en los ejemplos principales. Los únicos valores introducidos artificialmente fueron los del padding, identificados explícitamente mediante una máscara de validez. El mensaje central es que **cada eje debe tener un significado y las operaciones deben respetarlo**. Dos tensores pueden tener la misma forma y representar datos completamente distintos.

---

## Done with this section

Next up: **03 · Indexing and broadcasting real data** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)